In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [3]:
N=x_train.shape[0]
print(N)

60000


In [4]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
print(x_train.shape)
print(x_test.shape)

(60000, 28, 28, 1)
(10000, 28, 28, 1)


In [5]:
filters=np.random.rand(8,3,3,1).astype('float32') #3x3x1 필터 8개
filters_flatten=filters.reshape(8, -1).T #필터 flatten

x_train_padded=np.zeros((x_train.shape[0], x_train.shape[1]+2, x_train.shape[2]+2, x_train.shape[3]))
x_train_padded[:,1:-1,1:-1,:]=x_train

x_train_overlapped=np.zeros((x_train.shape[0], x_train.shape[1]*x_train.shape[2], 3*3*1))
for i in range(x_train.shape[1]):
    for j in range(x_train.shape[2]):
        patch = x_train_padded[:, i:i+3, j:j+3, :]
        x_train_overlapped[:, i*x_train.shape[2]+j, :] = patch.reshape(x_train.shape[0], -1)

print(x_train_overlapped.shape)
print(filters_flatten.shape)

(60000, 784, 9)
(9, 8)


In [6]:
cnn_output=np.dot(x_train_overlapped, filters_flatten)
print(cnn_output.shape)

(60000, 784, 8)


In [7]:
bias=np.zeros((8,), dtype='float32')
cnn_output += bias

In [8]:
#ReLU
cnn_output = np.maximum(0, cnn_output)

In [9]:
# FC layer
cnn_output_flatten=cnn_output.reshape(N, -1)
print(cnn_output_flatten.shape)

(60000, 6272)


In [10]:
W2=0.01 * np.random.randn(cnn_output_flatten.shape[1], 10).astype('float32')
b2=np.zeros((1,10), dtype='float32')

print(W2.shape)

(6272, 10)


In [11]:
output_score=np.dot(cnn_output_flatten,W2) + b2
print(output_score.shape)

(60000, 10)


In [12]:
# Softmax
exp_scores = np.exp(output_score)
probs= exp_scores/ np.sum(exp_scores, axis=1, keepdims=True)
print(probs.shape)

(60000, 10)


In [13]:
losses= -np.log(probs[np.arange(N), y_train])
data_loss=np.sum(losses,axis=0)/(N)
reg = 1e-3
regulation_loss1=0.5*reg*np.sum(W2*W2)
regulation_loss2=0.5*reg*np.sum(filters*filters)
total_loss=data_loss + regulation_loss1 + regulation_loss2
## 여기까지 feed forward


In [14]:
#gradient
dscore=probs
dscore[np.arange(N),y_train]-=1
dscore/=(N)
dW2=np.dot(cnn_output_flatten.T, dscore)
dW2+=reg*W2
db2=np.sum(dscore,axis=0,keepdims=True)


print(dscore.shape)
print(dW2.shape)
print(db2.shape)

(60000, 10)
(6272, 10)
(1, 10)


In [15]:
dcnn_output_flatten=np.dot(dscore,W2.T)





In [16]:
dcnn_output_flatten[cnn_output_flatten <= 0] = 0


In [ ]:
dfilter_flat=np.dot(x_train_overlapped.T, dcnn_output_flatten)


In [ ]:
dfilter_flat+=reg*filters


In [ ]:
db1=np.sum(dcnn_output_flatten, axis=0, keepdims=True)

In [ ]:
print(dcnn_output_flatten.shape)
print(dfilter_flat.shape)
print(db1.shape)

(60000, 784, 8)
